In [1]:
import json
import os
import glob
import subprocess
import torch
import sys
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import LlamaConfig, LlamaForCausalLM, AutoTokenizer
from transformers import get_cosine_schedule_with_warmup

c:\Users\ADMIN\.unsloth\studio\unsloth_studio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).
W0815 07:31:25.350000 16396 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
import json
import os

# ==========================================
# CONFIGURATION & HYPERPARAMETERS
# ==========================================
CONFIG_PATH = os.environ.get("CONFIG_PATH", "config/config_5k.json")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    _cfg = json.load(f)

print(json.dumps(_cfg, indent=4))

# Data & File Paths
DATA_FILE = _cfg["data_paths"]["data_file"]
CHECKPOINT_DIR = _cfg["data_paths"]["checkpoint_dir"]
GGUF_OUTPUT_PATH = _cfg["data_paths"]["gguf_output_path"]

# Tokenizer Constants
VOCAB_SIZE = _cfg["tokenizer"]["vocab_size"]
UNK_TOKEN = _cfg["tokenizer"]["unk_token"]
BOS_TOKEN = _cfg["tokenizer"]["bos_token"]
EOS_TOKEN = _cfg["tokenizer"]["eos_token"]
PAD_TOKEN = _cfg["tokenizer"]["pad_token"]
SPECIAL_TOKENS = [UNK_TOKEN, BOS_TOKEN, EOS_TOKEN, PAD_TOKEN]

# Model Architecture Hyperparameters
HIDDEN_SIZE = _cfg["model"]["hidden_size"]
INTERMEDIATE_SIZE = _cfg["model"]["intermediate_size"]
NUM_HIDDEN_LAYERS = _cfg["model"]["num_hidden_layers"]
NUM_ATTENTION_HEADS = _cfg["model"]["num_attention_heads"]
MAX_POSITION_EMBEDDINGS = _cfg["model"]["max_position_embeddings"]

# Dataset & DataLoader Hyperparameters
MAX_SEQUENCE_LENGTH = _cfg["dataset"]["max_sequence_length"]
IGNORE_INDEX = _cfg["dataset"]["ignore_index"]
BATCH_SIZE = _cfg["dataset"]["batch_size"]
ACCUMULATION_STEPS = _cfg["dataset"]["accumulation_steps"]
VAL_SPLIT = _cfg["dataset"]["val_split"]

# Regularization
ATTENTION_DROPOUT = _cfg["regularization"]["attention_dropout"]
WEIGHT_DECAY = _cfg["regularization"]["weight_decay"]

# Training Hyperparameters
LEARNING_RATE = _cfg["training"]["learning_rate"]
STEPS_PER_RUN = _cfg["training"]["steps_per_run"]
LOG_INTERVAL = _cfg["training"]["log_interval"]
SAVE_INTERVAL = _cfg["training"]["save_interval"]
TOTAL_TRAINING_STEPS = _cfg["training"]["total_training_steps"]
WARMUP_STEPS = _cfg["training"]["warmup_steps"]

# Evaluation & Generation Configuration
TEST_PROMPT = _cfg["evaluation"]["test_prompt"]
MAX_NEW_TOKENS = _cfg["evaluation"]["max_new_tokens"]

{
    "data_paths": {
        "data_file": "train_5k.jsonl",
        "checkpoint_dir": "./checkpoints",
        "gguf_output_path": "model.gguf"
    },
    "tokenizer": {
        "vocab_size": 6000,
        "unk_token": "<unk>",
        "bos_token": "<s>",
        "eos_token": "</s>",
        "pad_token": "<pad>"
    },
    "model": {
        "hidden_size": 256,
        "intermediate_size": 688,
        "num_hidden_layers": 6,
        "num_attention_heads": 8,
        "max_position_embeddings": 192
    },
    "dataset": {
        "max_sequence_length": 256,
        "ignore_index": -100,
        "batch_size": 8,
        "accumulation_steps": 4,
        "val_split": 200
    },
    "regularization": {
        "attention_dropout": 0.1,
        "weight_decay": 0.01
    },
    "training": {
        "learning_rate": 0.0005,
        "steps_per_run": 500,
        "log_interval": 10,
        "save_interval": 50,
        "total_training_steps": 3000,
        "warmup_steps": 300
    },
    "evalua

In [3]:
import random

train_path = "./data_split/train.jsonl"
val_path = "./data_split/val.jsonl"

if os.path.exists(train_path) and os.path.exists(val_path):
    print("Found cached train/val split, skipping re-split.")
    with open(train_path, 'r', encoding='utf-8') as f:
        train_lines = [line.rstrip("\n") for line in f]
    with open(val_path, 'r', encoding='utf-8') as f:
        val_lines = [line.rstrip("\n") for line in f]
else:
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        all_lines = [line.strip() for line in f if line.strip()]

    random.seed(42)
    random.shuffle(all_lines)

    val_lines = all_lines[:VAL_SPLIT]
    train_lines = all_lines[VAL_SPLIT:]

    os.makedirs("./data_split", exist_ok=True)
    with open(train_path, "w", encoding="utf-8") as f:
        f.write("\n".join(train_lines) + "\n")
    with open(val_path, "w", encoding="utf-8") as f:
        f.write("\n".join(val_lines) + "\n")

print(f"Train rows: {len(train_lines)} | Val rows: {len(val_lines)}")

Train rows: 4800 | Val rows: 200


In [4]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

tokenizer_hf_dir = "./tokenizer_hf"

if os.path.exists(os.path.join(tokenizer_hf_dir, "tokenizer.json")):
    print("Found cached tokenizer, loading from disk.")
    tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_hf_dir)
else:
    os.makedirs("./tokenizer_raw", exist_ok=True)

    raw_bpe = ByteLevelBPETokenizer()
    raw_bpe.train(
        files=["./data_split/train.jsonl"],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
    )
    raw_bpe.save_model("./tokenizer_raw")

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=raw_bpe,
        unk_token=UNK_TOKEN,
        bos_token=BOS_TOKEN,
        eos_token=EOS_TOKEN,
        pad_token=PAD_TOKEN,
    )

    os.makedirs(tokenizer_hf_dir, exist_ok=True)
    tokenizer.save_pretrained(tokenizer_hf_dir)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_hf_dir)

actual_vocab_size = len(tokenizer)
print(f"Tokenizer ready. Vocab size: {actual_vocab_size}")

Tokenizer ready. Vocab size: 6000


In [5]:
# ==========================================
# 2. Model & Config Setup
# ==========================================
config = LlamaConfig(
    vocab_size=actual_vocab_size,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    max_position_embeddings=MAX_POSITION_EMBEDDINGS,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    attention_dropout=ATTENTION_DROPOUT, 
)

model = LlamaForCausalLM(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Total Parameters: {total_params:,} (Trainable: {trainable_params:,})")

# ==========================================
# 3. Dataset & DataLoader
# ==========================================
class JSONLDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=MAX_SEQUENCE_LENGTH):
        self.examples = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                formatted_text = f"{tokenizer.eos_token}Input: {data['input']}\nOutput: {data['output']}{tokenizer.eos_token}"
                self.examples.append(formatted_text)
                
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        
        labels = input_ids.clone()
        labels[attention_mask == 0] = IGNORE_INDEX

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

dataset = JSONLDataset("./data_split/train.jsonl", tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = JSONLDataset("./data_split/val.jsonl", tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)  # <-- NEW

# ==========================================
# 4. Training Loop & Auto-Resume Setup
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

step = 0
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def get_latest_checkpoint(ckpt_dir):
    ckpts = glob.glob(os.path.join(ckpt_dir, "step_*"))
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: int(x.split("_")[-1]))
    return ckpts[-1]

Model Total Parameters: 7,818,496 (Trainable: 7,818,496)


In [6]:

latest_ckpt = get_latest_checkpoint(CHECKPOINT_DIR)

if latest_ckpt and os.path.exists(os.path.join(latest_ckpt, "training_state.pt")):
    print(f"Resuming training from checkpoint: {latest_ckpt}")
    
    model = LlamaForCausalLM.from_pretrained(latest_ckpt).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    train_state = torch.load(os.path.join(latest_ckpt, "training_state.pt"), map_location=device)
    optimizer.load_state_dict(train_state["optimizer_state"])
    step = train_state["step"]
    
    torch.set_rng_state(train_state["cpu_rng_state"].cpu())
    if torch.cuda.is_available() and train_state.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state(train_state["cuda_rng_state"].cpu())
        
    print(f"Successfully loaded state. Resuming at Step {step}\n")
else:
    print(f"No previous checkpoint found. Starting fresh training on {device}...\n")

# <--- DYNAMIC TARGET CALCULATION --->
target_step = step + STEPS_PER_RUN
print(f"Will run for {STEPS_PER_RUN} steps. Stopping at Step {target_step}.\n")

if target_step > TOTAL_TRAINING_STEPS:
    print(f"⚠️ target_step ({target_step}) exceeds TOTAL_TRAINING_STEPS ({TOTAL_TRAINING_STEPS}). "
          f"Capping at {TOTAL_TRAINING_STEPS}.")
    target_step = TOTAL_TRAINING_STEPS

# --- scheduler spans the FULL planned training run, not just this session ---
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS   # <-- fixed, not target_step
)

# --- restore scheduler position if resuming ---
if latest_ckpt and os.path.exists(os.path.join(latest_ckpt, "training_state.pt")):
    if "scheduler_state" in train_state:
        scheduler.load_state_dict(train_state["scheduler_state"])
    else:
        # old checkpoint without scheduler state — fast-forward manually
        for _ in range(step):
            scheduler.step()

model.train()
data_iter = iter(dataloader)
micro_step = 0   # <-- NEW: tracks forward/backward passes separately from optimizer steps

@torch.no_grad()
def compute_val_loss(model, val_dataloader, device):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        n_tokens = (labels != IGNORE_INDEX).sum().item()
        total_loss += out.loss.item() * n_tokens
        total_tokens += n_tokens
    model.train()
    return total_loss / max(total_tokens, 1)

import logging
import shutil
import time

log_path = os.path.join(CHECKPOINT_DIR, f"training_{int(time.time())}.log")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

logger = logging.getLogger("train_logger")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if cell is re-run

file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(console_handler)

logger.info(f"=== Training session started ===")
logger.info(f"Full schedule: warmup={WARMUP_STEPS}, total={TOTAL_TRAINING_STEPS}, "
            f"currently at step={step} ({step/TOTAL_TRAINING_STEPS:.1%} through decay)")

def cleanup_old_checkpoints(ckpt_dir, keep=2):
    ckpts = glob.glob(os.path.join(ckpt_dir, "step_*"))
    ckpts.sort(key=lambda x: int(x.split("_")[-1]))
    to_delete = ckpts[:-keep] if len(ckpts) > keep else []
    for ckpt_path in to_delete:
        shutil.rmtree(ckpt_path, ignore_errors=True)
        logger.info(f"Deleted old checkpoint: {ckpt_path}")

while step < target_step:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        batch = next(data_iter)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss / ACCUMULATION_STEPS
    loss.backward()

    micro_step += 1

    if micro_step % ACCUMULATION_STEPS == 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        step += 1

        if step % LOG_INTERVAL == 0 or step == 1:
            current_lr = optimizer.param_groups[0]["lr"]
            logger.info(f"Step {step}/{target_step} | Loss: {loss.item() * ACCUMULATION_STEPS:.4f} | LR: {current_lr:.6e}")

            val_loss = compute_val_loss(model, val_dataloader, device)
            logger.info(f"  >> Validation Loss: {val_loss:.4f}")

            model.eval()

            def test_gen(prompt, is_unseen=False):
                prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
                with torch.no_grad():
                    gen_tokens = model.generate(
                        prompt_ids,
                        max_new_tokens=MAX_NEW_TOKENS,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        do_sample=False
                    )
                generated_text = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
                label = "Unseen" if is_unseen else "Seen"
                logger.info(f"  --> {label} Generation @ step {step}:\n{generated_text}\n" + "-"*50)

            test_gen(TEST_PROMPT)
            model.train()

        if step % SAVE_INTERVAL == 0 or step == target_step:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"step_{step}")
            logger.info(f"Saving checkpoint to {ckpt_path}...")
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            train_state = {
                "step": step,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "cpu_rng_state": torch.get_rng_state(),
                "cuda_rng_state": torch.cuda.get_rng_state() if torch.cuda.is_available() else None,
            }
            torch.save(train_state, os.path.join(ckpt_path, "training_state.pt"))
            logger.info("Checkpoint saved successfully!\n")

            cleanup_old_checkpoints(CHECKPOINT_DIR, keep=2)   # <-- NEW: prune old checkpoints

logger.info("Training finished!")

=== Training session started ===
Full schedule: warmup=300, total=3000, currently at step=0 (0.0% through decay)


No previous checkpoint found. Starting fresh training on cuda...

Will run for 500 steps. Stopping at Step 500.



Step 1/500 | Loss: 8.8036 | LR: 1.666667e-06
  >> Validation Loss: 8.7982
  --> Seen Generation @ step 1:
Input: Translate to Vietnamese: Sorry, that question's not on here.
Output: Minrigrigrigrigrigrigrigrigrig Min Min Min Min MinrigBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạnBạn energymen Min Min Min Min Min MinuốcBạnuốcBạnuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốcuốc
--------------------------------------------------
Step 10/500 | Loss: 8.6354 | LR: 1.666667e-05
  >> Validation Loss: 8.6011
  --> Seen Generation @ step 10:
Input: Translate to Vietnamese: Sorry, that question's not on here.
Output:omửa causomửa caus caus caus administ administ administ administ administ administ administ administ administ administ administ administ administ thuậnuation thuậnuation thuậnuation thuậnuationuationuationuationuationuationuationuationuationuationuationuationuationuat

KeyboardInterrupt: 

In [7]:
# ==========================================
# 5. Convert Latest Checkpoint to GGUF
# ==========================================
latest_ckpt = get_latest_checkpoint(CHECKPOINT_DIR)

if latest_ckpt:
    print(f"\nConverting latest checkpoint ({latest_ckpt}) to GGUF format...")
    
    convert_script = "C:/Users/ADMIN/.unsloth/llama.cpp/convert_hf_to_gguf.py"
    
    if not os.path.exists(convert_script):
        print(f"Note: Ensure '{convert_script}' from llama.cpp repository is available in current directory.")
    
    cmd = [
        sys.executable, convert_script, 
        latest_ckpt,
        "--outfile", GGUF_OUTPUT_PATH,
        "--outtype", "f32",
    ]
    
    try:
        res = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Conversion complete! Saved to {GGUF_OUTPUT_PATH}")
    except FileNotFoundError:
        print(f"Failed to locate conversion script or python executable.")
    except subprocess.CalledProcessError as e:
        print(f"GGUF conversion failed with error:\n{e.stderr}")


Converting latest checkpoint (./checkpoints\step_200) to GGUF format...
Conversion complete! Saved to model.gguf


In [8]:
import subprocess
import sys

def kill_port(port):
    # Find the PID using the port
    result = subprocess.run(
        ["netstat", "-ano"],
        capture_output=True, text=True
    )

    lines = result.stdout.splitlines()
    pids = set()

    for line in lines:
        if f":{port}" in line and "LISTENING" in line:
            parts = line.split()
            pid = parts[-1]
            pids.add(pid)

    if not pids:
        print(f"No process found listening on port {port}.")
        return

    for pid in pids:
        print(f"Killing PID {pid} (using port {port})...")
        subprocess.run(["taskkill", "/PID", pid, "/F"])

kill_port(8080)

No process found listening on port 8080.


In [9]:
import subprocess
import os
import sys

def start_llama_server():
    log_file = open("llama_server.log", "w")

    process = subprocess.Popen(
        [
            "llama-server",
            "-m", "model.gguf",
            "-ngl", "99",
            "--host", "127.0.0.1",
            "--port", "8080"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,  # detaches from parent process group (Linux/Mac)
        close_fds=True,
    )

    # Save PID so you can stop it later
    with open("llama_server.pid", "w") as f:
        f.write(str(process.pid))

    print(f"llama-server started in background with PID {process.pid}")
    print("Logs: llama_server.log")
    return process.pid

start_llama_server()

llama-server started in background with PID 13652
Logs: llama_server.log


13652

In [10]:
import requests

prompt = "Input: Translate to Vietnamese: Though various cultures use different words for it, the concept of wondering “why?” is universal, and children around the world use it as a bootstrap for understanding the world.\nOutput:"

resp = requests.post(
    "http://127.0.0.1:8080/v1/completions",
    json={
        "model": "model.gguf",
        "prompt": prompt,
        "max_tokens": 128,
        "temperature": 0.1,
        "top_p": 1.0,
        "stream": False,
    },
    timeout=120,
)

print("STATUS:", resp.status_code)
print(resp.text)

if resp.ok:
    data = resp.json()
    print("\nOUTPUT:")
    print(data["choices"][0]["text"])

STATUS: 200
{"choices":[{"text":" Cput: Các là một một một một một một trong một một một trong một trong một một thành một trong một các trong một trong một là một trong một trong một trong một trong một.","index":0,"logprobs":null,"finish_reason":"stop"}],"created":1786754107,"model":"model.gguf","system_fingerprint":"b10182-afeebe103","object":"text_completion","usage":{"completion_tokens":42,"prompt_tokens":63,"total_tokens":105,"prompt_tokens_details":{"cached_tokens":0}},"id":"chatcmpl-bcLM7TY6EEIaSdXNshCEfEJfNahZPtEw","timings":{"cache_n":0,"prompt_n":63,"prompt_ms":147.306,"prompt_per_token_ms":2.3381904761904764,"prompt_per_second":427.6811535171683,"predicted_n":42,"predicted_ms":38.777,"predicted_per_token_ms":0.9232619047619048,"predicted_per_second":1083.1162802692318}}

OUTPUT:
 Cput: Các là một một một một một một trong một một một trong một trong một một thành một trong một các trong một trong một là một trong một trong một trong một trong một.
